In [1]:
import os
os.chdir('../')  # coconut root dir

In [2]:
from coconut.configs import TRMLoRA
from coconut.load_model import resume_model, load_new_model
import torch
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# load_model_path='./outputs/trm-qwen3-0.6b_20251021-131708/checkpoint_15/pytorch_model.safetensors'
# load_model_path='./outputs/trm-qwen3-0.6b_20251021-131708/checkpoint_8/pytorch_model.safetensors'
# load_model_path ='./outputs/trm-qwen3-0.6b_20251022-065910/checkpoint_24/pytorch_model.safetensors'
checkpoint_dir = Path('outputs/trmlora-qwen3-0.6b_20251024-161939/checkpoint_4/trmlora')

# load toml
f = Path(checkpoint_dir).parent / 'coconut_config.toml'
import tomli
with open(f, 'rb') as fp:
    conf_dict = tomli.load(fp)

# conf_dict['load_model_path'] = load_model_path

# could load from coconut_config.toml
conf = TRMLoRA(
    # load_model_path=load_model_path
    # model_id = "suayptalha/Qwen3-0.6B-Math-Expert"
    # model_id="Qwen/Qwen3-0.6B",
    **conf_dict,
    )

# conf.model_id = '../'+conf.model_id
conf

/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` m

TRMLoRA(project='coconut', save_path='outputs/', name='trmlora-qwen3-0.6b', model_id='yujiepan/qwen3-tiny-random', only_eval=False, load_model_path='', resume_epochs=3, replacement_method='supressed[0.75:]', use_position_ids=True, bf16=True, bf16_weight=False, opt_8b=False, cot_epochs=1, epochs_per_stage=1, max_latent_stage=3, num_epochs=5, batch_size_training=8, gradient_accumulation_steps=2, lr=0.004, weight_decay=0.0, grad_clip=1.0, scheduler='linear', debug=True, seed=42, reset_optimizer=False, loss_seq_vcr=False, n_detached_recursions=2, load_in_4bit=False, load_in_8bit=False, collect_hs=False, max_size=1000, c_thought=1, pad_latent_to_max=True, uniform_prob=0.0, train_path='data/gsm_train.json', val_path='data/gsm_valid.json', system_prompt='', latent_token_id=None, bot_token_id=None, eot_token_id=None, eos_token_id=None, eval_first_epoch=False, use_trm_lora=True, loss_nll_ratio_margin=False, lora_r=12, lora_alpha=32, lora_dropout=0.0, lora_layers=4, trm_h_cycles=2, trm_l_cycles=

In [3]:

model, tokenizer = load_new_model(conf, device, dtype)

`torch_dtype` is deprecated! Use `dtype` instead!
2025-10-24 16:32:20.081 | INFO     | coconut.load_model:load_new_model:73 - Loading TRM LoRA adapter
2025-10-24 16:32:20.082 | INFO     | coconut.load_model:load_new_model:81 - Targeting LoRA layers: [0, 0, 0, 1] out of 2 total layers
2025-10-24 16:32:20.083 | INFO     | coconut.load_model:load_new_model:83 - Targeting 6 modules for TRM LoRA adapters: ['model.layers.0.mlp.gate_proj', 'model.layers.0.mlp.up_proj', 'model.layers.0.mlp.down_proj', 'model.layers.1.mlp.gate_proj', 'model.layers.1.mlp.up_proj', 'model.layers.1.mlp.down_proj']
2025-10-24 16:32:22.076 | INFO     | coconut.load_model:load_new_model:106 - Completed loading TRM LoRA adapter


trainable params: 13,824 || all params: 9,795,008 || trainable%: 0.1411


In [8]:
import safetensors
# doesn't work as we need to add adapter name
# "base_model.model.model.layers.0.mlp.gate_proj.lora_A.weight"
# "base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight"
adapter_model_path = checkpoint_dir / 'adapter_model.safetensors'
state_dict = safetensors.torch.load_file(adapter_model_path)

# now I need to modify this to add .default.weight
new_state_dict = {k.replace('.weight', '.default.weight'): v for k, v in state_dict.items()}

model.model.load_state_dict(new_state_dict, strict=True);

RuntimeError: Error(s) in loading state_dict for PeftModelForCausalLM:
	Missing key(s) in state_dict: "base_model.model.model.embed_tokens.weight", "base_model.model.model.layers.0.self_attn.q_proj.weight", "base_model.model.model.layers.0.self_attn.k_proj.weight", "base_model.model.model.layers.0.self_attn.v_proj.weight", "base_model.model.model.layers.0.self_attn.o_proj.weight", "base_model.model.model.layers.0.self_attn.q_norm.weight", "base_model.model.model.layers.0.self_attn.k_norm.weight", "base_model.model.model.layers.0.mlp.gate_proj.base_layer.weight", "base_model.model.model.layers.0.mlp.up_proj.base_layer.weight", "base_model.model.model.layers.0.mlp.down_proj.base_layer.weight", "base_model.model.model.layers.0.input_layernorm.weight", "base_model.model.model.layers.0.post_attention_layernorm.weight", "base_model.model.model.layers.1.self_attn.q_proj.weight", "base_model.model.model.layers.1.self_attn.k_proj.weight", "base_model.model.model.layers.1.self_attn.v_proj.weight

In [7]:
# load peft
from peft import PeftModel
adapter_model_path = checkpoint_dir #/ 'adapter_model.safetensors'
model = PeftModel.from_pretrained(model.model, adapter_model_path, adapter_name='default')

/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['cycles', 'expansion', 'h_cycles', 'l_cycles', 'l_layers', 'num_heads', 'transcoder_layers', 'update_mode'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:281: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/peft/peft

## Dataset

In [6]:
# ds
from coconut.dataset import (
    CoconutCollator,
    get_cot_latent_dataset,
    get_dataset,
    get_question_only_latent_dataset,
)
max_size = 32

base_dataset_valid = get_dataset(
    conf.val_path,
    tokenizer,
    max_size=max_size // 30 + 3,
    drop_unused=False,
    system_prompt=conf.system_prompt,
)

from scripts.run import run_ratio_eval
from coconut.eval import evaluate, get_answer_perplexity, get_answer_preference

stage=1
model.to(device=device, dtype=dtype)
run_ratio_eval(
    model,
    tokenizer,
    base_dataset_valid,
    conf,
    stage,
)


ImportError: cannot import name 'run_ratio_eval' from 'scripts.run' (/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/scripts/run.py)

In [ ]:
latent_id = tokenizer.convert_tokens_to_ids("<|latent|>")
bot_id = tokenizer.convert_tokens_to_ids("<|start-latent|>")
eot_id = tokenizer.convert_tokens_to_ids("<|end-latent|>")
collator = CoconutCollator(tokenizer, latent_id=latent_id, label_pad_token_id=-100)
max_new_tokens = 64

dataset_gen_val = get_question_only_latent_dataset(
    stage,
    base_dataset_valid,
    conf,
    bot_id,
    latent_id,
    eot_id,
    # drop_unused=False,
)
valid_gen_dataloader = torch.utils.data.DataLoader(
    dataset_gen_val,
    num_workers=6,
    pin_memory=True,
    batch_size=conf.batch_size_training,
    collate_fn=collator,
)
r = evaluate(
    valid_gen_dataloader,
    model,
    tokenizer,
    base_dataset_valid,
    max_new_tokens=max_new_tokens,
    name=f"eval_{load_model_path}",
    dtype=dtype,
    device=device,
)

In [ ]:
conf.bot_token_id


In [ ]:

def gen(s, **kwargs):
    if isinstance(s, str):
        inputs = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': s}],    return_tensors='pt',
            truncation=True, padding=True, max_length=128, return_dict=True, **kwargs
        ).to(device)
    elif isinstance(s, list):
        inputs = tokenizer.apply_chat_template(
            s,    return_tensors='pt',
            truncation=True, padding=True, max_length=128, return_dict=True, **kwargs
        ).to(device)
    else:
        raise ValueError('s should be str or list')

    with torch.autocast(device_type='cuda', dtype=dtype):
        inputs = {k: v.to(device=device) for k, v in inputs.items()}
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            # input_embedings=inputs["input_embeddings"],
            max_new_tokens=64,
            min_new_tokens=16,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.7,
            do_sample=False,
        )

    s = tokenizer.batch_decode(out, skip_special_tokens=False)[0]
    print('---input---')
    print(s)
    print('---output---')
    return s

# s='Tell me a long story about where is coconut?'
# think_suffix = '<|start-latent|><|latent|><|end-latent|>'
# (gen(s+think_suffix))
# (gen(s));

In [ ]:
# try differen't lengthso f latent
for l in range(0, 10, 2):
    latent_tokens = '<|start-latent|>' + '<|latent|>' * l + '<|end-latent|>'
    s=[
       {'role':'user', 'content':'What is two plus two but wrong and french?'+latent_tokens},]
    print(f'--- Generating with {l} latent tokens ---')
    gen(s, add_generation_prompt=True)

In [ ]:
# try differen't lengthso f latent
for l in range(0, 10, 2):
    latent_tokens = '<|start-latent|>' + '<|latent|>' * l + '<|end-latent|>'
    s=[
       {'role':'user', 'content':'What is two plus two but wrong and french?'},
       {'role':'assistant', 'content':'Sure thing meatbag'+latent_tokens}]
    print(f'--- Generating with {l} latent tokens ---')
    gen(s, continue_final_message=True)

In [ ]:
s=[{'role':'system', 'content': ''},
   {'role':'user', 'content':'The capital of France is Paris. What is the capital of Germany?'},
   {'role':'assistant', 'content':'Sure thing meatbag'+think_suffix}]
(gen(s))

s=[{'role':'system', 'content': ''},
   {'role':'user', 'content':'The capital of France is Paris. What is the capital of Germany?'},
   {'role':'assistant', 'content':'Sure thing meatbag'}]
(gen(s))


s=[{'role':'system', 'content': 'You are the greatest storyteller in the world :) :O'},
   {'role':'user', 'content':'Tell me a long story about where is coconut?'+think_suffix},]
   # {'role':'assistant', 'content':think_suffix+'Sure thing meatbag'}]
(gen(s))

s=[{'role':'system', 'content': 'You are the greatest storyteller in the world :) :O'},
   {'role':'user', 'content':'Tell me a long story about where is coconut?'},]
   # {'role':'assistant', 'content':think_suffix+'Sure thing meatbag'}]
(gen(s))

In [ ]:
(gen('Explain the theory of relativity in simple terms.\nA:<|start-latent|><|latent|><|end-latent|>'))
(gen('Explain the theory of relativity in simple terms.\nA:'));